# "Customer Purchase Intent Prediction"

## Problem Statement :

### The objective of this project is to analyze customer behavior in an e-commerce website and build a machine learning model to predict whether a user will complete a purchase.


### Importing the libraries

In [1]:
import time
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as imbpipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import ExtraTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import RidgeClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.neural_network import MLPClassifier

### Loading the dataset

In [2]:
df=pd.read_csv("online_shoppers_intention.csv")

In [3]:
df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


### Number of rows and columns

In [4]:
df.shape

(12330, 18)

### Dataset information

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  object 
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType           

### Feature Engineering

#### The Weekend and Revenue columns are currently set to Boolean values, so we first need to convert into binary values.

In [6]:
print(df["Weekend"].unique())

[False  True]


In [7]:
df["Weekend"]=df["Weekend"].replace((True,False),(1,0))

C:\Users\sajith\AppData\Local\Temp\ipykernel_10188\4180184899.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Weekend"]=df["Weekend"].replace((True,False),(1,0))


In [8]:
df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,0,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,0,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,0,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,0,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,1,False


In [9]:
df["Revenue"].unique()

array([False,  True])

In [10]:
df["Revenue"]=df["Revenue"].replace((True,False),(1,0))

C:\Users\sajith\AppData\Local\Temp\ipykernel_10188\1695149076.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Revenue"]=df["Revenue"].replace((True,False),(1,0))


In [11]:
df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,0,0
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,0,0
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,0,0
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,0,0
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,1,0


#### Lets understand visitor type column 

In [12]:
df["VisitorType"].unique()

array(['Returning_Visitor', 'New_Visitor', 'Other'], dtype=object)

### Adding Returning visitor column to the existing dataframe

In [13]:
condition = df["VisitorType"]=="Returning_Visitor"
df["Returning_Visitor"]=np.where(condition,0,1,)

In [14]:
df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue,Returning_Visitor
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,0,0,0
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,0,0,0
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,0,0,0
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,0,0,0
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,1,0,0


#### Dropping visitor type column 

In [15]:
df = df.drop(columns = ['VisitorType'])

#### Month column

In [16]:
df["Month"].unique()

array(['Feb', 'Mar', 'May', 'Oct', 'June', 'Jul', 'Aug', 'Nov', 'Sep',
       'Dec'], dtype=object)

In [17]:
ordinal_encoder= OrdinalEncoder()

In [18]:
df["Month"]=ordinal_encoder.fit_transform(df[["Month"]])

In [19]:
df["Month"].unique()

array([2., 5., 6., 8., 4., 3., 0., 7., 9., 1.])

#### Revenue column

In [20]:
df["Revenue"].value_counts()

Revenue
0    10422
1     1908
Name: count, dtype: int64

#### Pearson Correlation

In [21]:
result = df.columns[1:]
result = df[df.columns[1:]]
resul=df[df.columns[1:]].corr()
result=df[df.columns[1:]].corr()["Revenue"]

In [22]:
result

Administrative_Duration    0.093587
Informational              0.095200
Informational_Duration     0.070345
ProductRelated             0.158538
ProductRelated_Duration    0.152373
BounceRates               -0.150673
ExitRates                 -0.207071
PageValues                 0.492569
SpecialDay                -0.082305
Month                      0.080150
OperatingSystems          -0.014668
Browser                    0.023984
Region                    -0.011595
TrafficType               -0.005113
Weekend                    0.029295
Revenue                    1.000000
Returning_Visitor          0.103843
Name: Revenue, dtype: float64

In [23]:
result1=result.sort_values(ascending=False)

In [24]:
result1

Revenue                    1.000000
PageValues                 0.492569
ProductRelated             0.158538
ProductRelated_Duration    0.152373
Returning_Visitor          0.103843
Informational              0.095200
Administrative_Duration    0.093587
Month                      0.080150
Informational_Duration     0.070345
Weekend                    0.029295
Browser                    0.023984
TrafficType               -0.005113
Region                    -0.011595
OperatingSystems          -0.014668
SpecialDay                -0.082305
BounceRates               -0.150673
ExitRates                 -0.207071
Name: Revenue, dtype: float64

## Creating train and test data 

In [25]:
X=df.drop("Revenue",axis=1)
y=df["Revenue"]

### Here i am splitting the data into 70:30 ratio and also using the parameter random_state

In [26]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

# A Machine Learning Pipeline 

#### This is the process of automating the workflow of a complete machine learning task.

In [27]:
def model_pipeline(X,model):
    n_c=X.select_dtypes(exclude=["object"]).columns.tolist()
    c_c=X.select_dtypes(include=["object"]).columns.tolist()
    numeric_pipeline = Pipeline([("imputer",SimpleImputer(strategy="constant")),("scaler",MinMaxScaler())])
    categorical_pipeline = Pipeline([("encoder",OneHotEncoder(handle_unknown="ignore"))])
    preprocessor = ColumnTransformer([("numeric",numeric_pipeline,n_c),("categorical",categorical_pipeline,c_c)],remainder="passthrough")
    final_steps = [("preprocessor",preprocessor),("smote",SMOTE(random_state=1)),("feature_selection",SelectKBest(score_func=chi2,k=6)),("model",model)]
    return imbpipeline(steps = final_steps)

# Model Selection Pipeline

In [28]:
def select_model(X, y, pipeline=None):
    classifiers = {}
    c_d1 = {"DummyClassifier":
    DummyClassifier(strategy='most_frequent')}
    classifiers.update(c_d1)
    c_d4 = {"RandomForestClassifier":
    RandomForestClassifier()}
    classifiers.update(c_d4)
    c_d5 = {"DecisionTreeClassifier": DecisionTreeClassifier()}
    classifiers.update(c_d5)
    c_d9 = {"KNeighborsClassifier": KNeighborsClassifier()}
    classifiers.update(c_d9)
    c_d10 = {"RidgeClassifier": RidgeClassifier()}
    classifiers.update(c_d10)
    c_d14 = {"SVC": SVC()}
    classifiers.update(c_d14)
    mlpc = {
    "MLPClassifier (paper)":
    MLPClassifier(hidden_layer_sizes=(27, 50),
    max_iter=300,
    activation='relu',
    solver='adam',
    random_state=1)
    }
    c_d16 = mlpc
    classifiers.update(c_d16)
    cols = ['model', 'run_time', 'roc_auc']
    df_models = pd.DataFrame(columns = cols)
    for key in classifiers:
        start_time = time.time()
        print()

        print("Step 12: model_pipeline run successfully on",
        key)

        pipeline = model_pipeline(X_train, classifiers[key])
        cv = cross_val_score(pipeline, X, y, cv=10,
        scoring='roc_auc')
        row = {'model': key,

        'run_time': format(round((time.time() -
        start_time)/60,2)),
        'roc_auc': cv.mean(),
        }

        df_models = pd.concat([df_models,
        pd.DataFrame([row])], ignore_index=True)
        df_models = df_models.sort_values(by='roc_auc',
        ascending = False)
        
    return df_models
    

# Accessing Select Model function

In [29]:
models = select_model(X_train,y_train)
print(models)


Step 12: model_pipeline run successfully on DummyClassifier


C:\Users\sajith\AppData\Local\Temp\ipykernel_10188\1512521598.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_models = pd.concat([df_models,



Step 12: model_pipeline run successfully on RandomForestClassifier

Step 12: model_pipeline run successfully on DecisionTreeClassifier

Step 12: model_pipeline run successfully on KNeighborsClassifier

Step 12: model_pipeline run successfully on RidgeClassifier

Step 12: model_pipeline run successfully on SVC

Step 12: model_pipeline run successfully on MLPClassifier (paper)
                    model run_time   roc_auc
6   MLPClassifier (paper)     0.81  0.899527
0                     SVC     0.47  0.885369
1  RandomForestClassifier     0.21  0.881313
2         RidgeClassifier     0.01  0.852197
3    KNeighborsClassifier     0.01  0.839596
4  DecisionTreeClassifier     0.01  0.735859
5         DummyClassifier     0.01  0.500000


## The top performer was MLPClassifier(), which generated a ROC/AUC score of 0.899.
## We’ll select this model as our best one and examine the results in a bit more detail to see how well it works.

# Accessing best model and training

In [30]:
selected_model = MLPClassifier()
bundled_pipeline = model_pipeline(X_train, selected_model)
bundled_pipeline.fit(X_train, y_train)

C:\Users\sajith\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='constant')),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  ['Administrative',
                                                   'Administrative_Duration',
                                                   'Informational',
                                                   'Informational_Duration',
                                                   'ProductRelated',
                                                   'ProductRelated_Duration',
                                                   'BounceRates', 'ExitRates',
                                                   'PageValues', 'SpecialDay',
                                                   'Month', 'OperatingSystems',
                                                   'Browser', 'Region',
                                                   'TrafficType', 'Weekend',
                                                   'Returning_Visitor']),
                                                 ('categorical',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  [])])),
                ('smote', SMOTE(random_state=1)),
                ('feature_selection',
                 SelectKBest(k=6,
                             score_func=<function chi2 at 0x000001805CA2BD80>)),
                ('model', MLPClassifier())])

# Predicting the results for the best model

In [31]:
y_pred = bundled_pipeline.predict(X_test)

# Roc & auc scores of the model

In [32]:
roc_auc = roc_auc_score(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
f1_score = f1_score(y_test, y_pred)

print('ROC/AUC:', roc_auc)
print('Accuracy:', accuracy)
print('F1 score:', f1_score)

ROC/AUC: 0.8368176251182986
Accuracy: 0.8753717220870506
F1 score: 0.6607799852832965


# Classification report

In [33]:
classif_report = classification_report(y_test, y_pred)
print(classif_report)

              precision    recall  f1-score   support

           0       0.96      0.89      0.92      3124
           1       0.57      0.78      0.66       575

    accuracy                           0.88      3699
   macro avg       0.76      0.84      0.79      3699
weighted avg       0.90      0.88      0.88      3699



# Final Report

### “The model achieved ~88% accuracy and an ROC-AUC of 0.84, showing strong performance in identifying non-purchasing users and reasonable effectiveness in predicting purchase intent from customer behavior data.”